In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.bronze.gho_raw")

In [0]:
sem_extras = bronze.Dim2.isNull() & bronze.Dim3.isNull()

indicator_filter = (
    ((bronze.IndicatorCode == "GHED_GGHE-D_pc_US_SHA2011") & bronze.Dim1.isNull() & sem_extras) |
    ((bronze.IndicatorCode == "HIV_0000000026") & bronze.Dim1.isNull() & sem_extras) |
    ((bronze.IndicatorCode == "MDG_0000000029") & bronze.Dim1.isNull() & sem_extras) |
    ((bronze.IndicatorCode == "NUTRITION_ANT_HAZ_NE2") & bronze.Dim1.isNull() & sem_extras) |

    ((bronze.IndicatorCode == "M_Est_smk_curr_std") & (bronze.Dim1 == "SEX_BTSX") & sem_extras) |
    ((bronze.IndicatorCode == "WHOSIS_000001") & (bronze.Dim1 == "SEX_BTSX") & sem_extras) |
    ((bronze.IndicatorCode == "WHOSIS_000002") & (bronze.Dim1 == "SEX_BTSX") & sem_extras) |

    ((bronze.IndicatorCode == "BP_04") & (bronze.Dim1 == "SEX_BTSX")
     & (bronze.Dim2 == "AGEGROUP_YEARS30-79") & bronze.Dim3.isNull()) |
    ((bronze.IndicatorCode == "NCDMORT3070") & (bronze.Dim1 == "SEX_BTSX")
     & (bronze.Dim2 == "AGEGROUP_YEARS30-69") & bronze.Dim3.isNull()) |
    ((bronze.IndicatorCode == "NCD_BMI_30C") & (bronze.Dim1 == "SEX_BTSX")
     & (bronze.Dim2 == "AGEGROUP_YEARS18-PLUS") & bronze.Dim3.isNull()) |
    ((bronze.IndicatorCode == "NCD_GLUC_04") & (bronze.Dim1 == "SEX_BTSX")
     & (bronze.Dim2 == "AGEGROUP_YEARS18-PLUS") & bronze.Dim3.isNull()) |

    ((bronze.IndicatorCode == "NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV")
     & (bronze.Dim1 == "SEX_FMLE")
     & (bronze.Dim2 == "PREGNANCYSTATUS_TOTAL")
     & bronze.Dim3.isNull()) |

    ((bronze.IndicatorCode == "SDGPM25") & (bronze.Dim1 == "RESIDENCEAREATYPE_TOTL") & sem_extras) |
    ((bronze.IndicatorCode == "WSH_WATER_SAFELY_MANAGED") & (bronze.Dim1 == "RESIDENCEAREATYPE_TOTL") & sem_extras) |
    ((bronze.IndicatorCode == "WSH_SANITATION_SAFELY_MANAGED") & (bronze.Dim1 == "RESIDENCEAREATYPE_TOTL") & sem_extras)
)

selected = (
    bronze
    .filter(F.col("SpatialDimType") == "COUNTRY")
    .filter(indicator_filter)
)

rejected = selected.filter(F.col("NumericValue").isNull())

silver = (
    selected
    .filter(F.col("NumericValue").isNotNull())
    .select(
        F.col("SpatialDim").alias("country_code"),
        F.col("ParentLocationCode").alias("region"),
        F.col("TimeDim").cast("int").alias("year"),
        F.col("IndicatorCode").alias("indicator_code"),
        F.col("NumericValue").cast("double").alias("numeric_value")
    )
)

In [0]:
assert silver.count() > 0, "silver is empty"

required = ["country_code", "region", "year", "indicator_code", "numeric_value"]
row = silver.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in required]
).first()
failures = {c: n for c, n in row.asDict().items() if n}
assert not failures, f"Nulls in required columns: {failures}"

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
(silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("workspace.silver.gho_clean"))
(rejected.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable("workspace.silver.gho_rejected"))

In [0]:
n_silver = spark.table("workspace.silver.gho_clean").count()
n_rejected = spark.table("workspace.silver.gho_rejected").count()
assert n_silver + n_rejected == selected.count(), "rows lost between bronze and silver"